# HGF Parameter Explorer

Interactive dashboard for exploring how HGF parameters affect belief
trajectories, learning rates, and choice probabilities.

**Two-part layout:**
1. **Model schematic** (static) — see *where* each parameter lives in the hierarchical model
2. **Behavioral panels** (interactive) — drag sliders to see *what* each parameter does

**Usage:** Run all cells top-to-bottom. The model schematic renders first,
then the interactive explorer warms up JAX (~5-10s). After that, slider
changes update the plots within ~2 seconds.

### Related notebooks

- **`quickstart_hgf.ipynb`** — End-to-end pipeline: simulate, fit, recover, compare
- **`design_experiment.ipynb`** — Power analysis: find the N you need before collecting data

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

%matplotlib widget
import matplotlib
matplotlib.rcParams["figure.dpi"] = 100

## Part 1: Model Structure

The schematic below shows the HGF's hierarchical architecture. Each
parameter label (gold boxes) corresponds to a slider in the interactive
panel below.

- **Solid blue arrows** = value coupling (prediction / prediction error)
- **Dashed coral arrows** = volatility coupling ($\kappa$ scales this)
- **Dotted purple arrows** = belief → response model ($\beta$, $\zeta$)

Toggle between 2-level and 3-level to see how the volatility node ($x_3$)
adds a meta-learning layer.

In [ ]:
import matplotlib.pyplot as plt
from prl_hgf.viz.model_schematic import plot_hgf_schematic

# 3-level model (default) — change to model_level=2 to see the simpler model
fig = plot_hgf_schematic(model_level=3, task="pick_best_cue")
plt.show()

# Uncomment to see the 2-level version side by side:
# fig2 = plot_hgf_schematic(model_level=2, task="pick_best_cue")
# plt.show()

## Part 2: Interactive Parameter Explorer

Now drag the sliders to see how each parameter affects the agent's behavior.
The model schematic above shows you *where* the parameter lives; the panels
below show you *what* it does.

**Tip:** Start with the presets (healthy baseline → post-psilocybin) to see
a clinically meaningful parameter shift, then fine-tune individual sliders.

In [ ]:
from prl_hgf.gui import ParamExplorer

explorer = ParamExplorer()

In [ ]:
explorer.display()

## Parameter Guide

| Parameter | Symbol | Effect |
|-----------|--------|--------|
| omega_2   | omega2  | Tonic volatility — higher = faster learning, more noise-sensitive |
| omega_3   | omega3  | Meta-volatility — how quickly volatility estimate changes (3-level only) |
| kappa     | kappa   | Coupling strength — how much volatility drives learning rate (3-level only) |
| beta      | beta    | Inverse temperature — higher = more exploitative, lower = more random |
| zeta      | zeta    | Stickiness — positive = perseverate on previous choice |
| mu_1_0    | mu1(0)  | Initial belief about P(reward) for all cues |
| mu_3_0    | mu3(0)  | Initial volatility prior mean (3-level only) |

## Preset Profiles

- **healthy baseline:** omega2=-3.0, kappa=1.0, beta=2.5 — balanced learning
- **post-concussion:** omega2=-4.0, kappa=0.8, beta=2.0 — slower learning, more perseveration
- **post-psilocybin:** omega2=-2.5, kappa=1.1, beta=2.0 — faster learning, increased coupling

Note: Belief trajectories are computed on a single simulated session (seed=0)
with fixed choices/rewards. The forward pass uses batch input_data (not
trial-by-trial simulation) for interactive speed.